In [1]:
from crewai.flow.flow import Flow, start, listen,router

from crewai.flow import (
    HumanFeedbackProvider,
    HumanFeedbackPending,
    PendingFeedbackContext,
    HumanFeedbackResult,
    human_feedback
)
from pydantic import BaseModel
from crewai import Agent, Task, Crew,LLM
from typing import Optional


import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
llm = LLM(
    model = "groq/llama-3.3-70b-versatile"
)

In [3]:
class ContentState(BaseModel):
    topic: str = ""
    draft: str = ""
    feedback: Optional[str] = None
    revision_count: int = 0
    approved: bool = False

In [4]:
MAX_ITERATIONS = 3

In [7]:
class ContentApprovalFlow(Flow[ContentState]):

    # ── Step 1: Entry point + Re-entry point ──
    @start("retry")
    def get_topic(self):
        print(f"\n📝 Topic: {self.state.topic}\n")
        return self.state.topic

    # ── Step 2: Generate draft ─────────────────
    @listen(get_topic)
    def generate_draft(self, topic):
        self.state.draft = f"# {topic}\n\nDraft about {topic}..."
        # In real use replace with LLM/Crew call
        return self.state.draft

    # ── Step 3: HITL node ONLY ─────────────────
    @listen(generate_draft)
    @human_feedback(
        message="Review this draft. Type 'approved' or give revision feedback:",
        emit=["approved", "needs_revision"],
        llm=llm,
        default_outcome="needs_revision",
    )
    def review_draft(self, draft):
        return f"TOPIC: {self.state.topic}\n\nDRAFT:\n{draft}"

    # ── Step 4: Router — decision maker ────────
    @router(review_draft)
    def route_feedback(self, result):    # ← plain string

        if result == "approved":         # ✅ correct for console
            return "complete"

        self.state.feedback = result     # ✅ result IS the feedback string
        self.state.revision_count += 1

        print("count is",self.state.revision_count)

        if self.state.revision_count >= MAX_ITERATIONS:
            return "max_retry_exceeded"

        return "retry"


    # ── Step 5: Approved ───────────────────────
    @listen("complete")
    def publish(self):
        self.state.approved = True
        print(f"\n✅ Published!\n")
        print(self.state.draft)


    # ── Step 6: Max retry exceeded ─────────────
    @listen("max_retry_exceeded")
    def on_max_retry(self):
        print(f"\n⚠ Max revisions reached. Saving last version.\n")
        self.state.approved = True
        print(self.state.draft)


In [9]:
flow = ContentApprovalFlow()
flow.kickoff(inputs={"topic": "What is Generative AI"})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ContentApprovalFlow                                                                                      │
│  ID: af923fec-7cf8-432e-9d63-f5c9283ca371                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ContentApprovalFlow                                                                                      │
│  ID: af923fec-7cf8-432e-9d63-f5c9283ca371                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: af923fec-7cf8-432e-9d63-f5c9283ca371

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: get_topic                                                                                              │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📝 Topic: What is Generative AI



...

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: route_feedback                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: route_feedback                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ContentApprovalFlow                                                                                      │
│  ID: af923fec-7cf8-432e-9d63-f5c9283ca371                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯